In [0]:
bronze_path = "abfss://bronze@stfintechlake.dfs.core.windows.net/transactions/*/*/*/*.json"

df = spark.read.json(bronze_path)

print("Total rows:", df.count())
df.printSchema()
display(df.limit(5))

Total rows: 101000
root
 |-- amount: double (nullable = true)
 |-- currency: string (nullable = true)
 |-- customer_email: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- description: string (nullable = true)
 |-- source_account_iban: string (nullable = true)
 |-- status: string (nullable = true)
 |-- target_account_iban: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- transaction_id: string (nullable = true)



amount,currency,customer_email,customer_id,customer_name,description,source_account_iban,status,target_account_iban,timestamp,transaction_id
-235.06,EUR,lisalee@example.com,C00394,Christy Harris,Transaction reversal,GB75OUTD21416141509377,COMPLETED,GB58HZBH88401677692191,2026-10-03 01:40:06,de02a99d-5ebb-4062-a911-22d64d558733
-679.17,QAR,lisa23@example.org,C00884,Carrie Hall,Restaurant,GB09GIWZ20284719017222,COMPLETED,GB67DEDU76208012893482,2026-10-03 12:01:19,e9a8b870-8713-47b5-a372-847a54d4371d
-497.31,EUR,omcdonald@example.net,C01350,Matthew Lawrence,Restaurant,GB69OZSG95795519247411,COMPLETED,GB22WYLB64103184173265,2026-10-03 10:19:56,1cf67904-9059-40c1-8242-39632fa5c283
-462.66,QAR,tylerchristine@example.com,C01252,Frank Martinez,Restaurant,GB29IKHI96551564196789,PENDING,GB92NIJV01989289143535,2026-10-03 16:34:52,6f16611b-8335-44b0-89bb-25a985312921
87.61,USD,samanthaherrera@example.com,C01710,Dr. Jason Foster,Cash deposit,GB05KUKA09159567832395,COMPLETED,GB85NLEW08313001618321,2026-10-03 17:07:24,0f881d3d-c877-4027-92ba-8795ea6acad8


In [0]:
from pyspark.sql import functions as F

ACCOUNT = "stfintechlake"
bronze_path = f"abfss://bronze@{ACCOUNT}.dfs.core.windows.net/transactions/*/*/*/*.json"

raw_df = spark.read.json(bronze_path)

print("Total rows:", raw_df.count())
print("Unique transaction_id:", raw_df.select("transaction_id").distinct().count())
print("Missing description:", raw_df.filter(F.col("description").isNull()).count())

Total rows: 101000
Unique transaction_id: 100000
Missing description: 505


In [0]:
dbutils.widgets.text("salt", "be9d96ebae394bb79886c1512cbc3098")
SALT = dbutils.widgets.get("salt")
assert SALT != "", "Upar widget box mein salt paste karo"

clean_df = (
    raw_df
    .dropDuplicates(["transaction_id"])
    .withColumn("amount", F.col("amount").cast("decimal(18,2)"))
    .withColumn("transaction_ts", F.to_timestamp("timestamp"))
    .withColumn("date_key", F.date_format("transaction_ts", "yyyyMMdd").cast("int"))
    .withColumn("description", F.coalesce(F.col("description"), F.lit("UNKNOWN")))
    .withColumn("direction", F.when(F.col("amount") > 0, "INFLOW").otherwise("OUTFLOW"))
    .withColumn("category",
        F.when(F.lower("description").contains("salary"), "SALARY")
         .when(F.lower("description").contains("sepa"), "TRANSFER")
         .when(F.lower("description").contains("reversal"), "REVERSAL")
         .otherwise("GENERAL"))
    .withColumn("customer_id_hashed", F.sha2(F.concat(F.col("customer_id"), F.lit(SALT)), 256))
    .drop("customer_id", "customer_name", "customer_email",
          "source_account_iban", "target_account_iban", "timestamp")
)

print("Rows after cleaning:", clean_df.count())
display(clean_df.limit(5))

Rows after cleaning: 100000


amount,currency,description,status,transaction_id,transaction_ts,date_key,direction,category,customer_id_hashed
-185.22,QAR,Utility bill,COMPLETED,aa3e4fbe-4b6f-4984-83c8-b511ce3fcdd9,2026-10-03T12:17:47.000Z,20261003,OUTFLOW,GENERAL,c673e9c51b6c60be27c69c05df8ac1ba1fefe37f7c7c34c9d277be6deeeb0a41
-166.39,EUR,Rent,COMPLETED,8dc244ad-255b-4e39-b372-69aac12b896c,2026-10-03T11:24:24.000Z,20261003,OUTFLOW,GENERAL,201f8ecef070f82bff7393c572ed55bee49e743b33dd2380f209bbfb7d73f2fc
-122.41,USD,Grocery,COMPLETED,2d6f135f-3e6f-47eb-9b75-acc0cdf2fc00,2026-10-03T02:08:20.000Z,20261003,OUTFLOW,GENERAL,7bc5db0015607984525cf12bcc232a87e6cfcdd6761db61049ecbe4d445f0879
-62.84,QAR,SEPA transfer sent,COMPLETED,481f1532-3d11-4e87-83e6-e81f15763ccd,2026-10-03T19:02:09.000Z,20261003,OUTFLOW,TRANSFER,0fa8f380da95f885bc3662a1adbb80bfe489a4ea2b794702d40812e2c6ade905
-612.48,USD,Transaction reversal,FAILED,187a67a5-d01b-40ef-9c5a-fb1e78941f27,2026-10-03T00:47:28.000Z,20261003,OUTFLOW,REVERSAL,6edca0af5b010773c96542a8f8db15c85c3bc398246ef8d761be5753504e5b6a


In [0]:
from delta.tables import DeltaTable

silver_path = f"abfss://silver@{ACCOUNT}.dfs.core.windows.net/transactions_delta"

if not DeltaTable.isDeltaTable(spark, silver_path):
    clean_df.write.format("delta").mode("overwrite").save(silver_path)
    print("1st attempt:")
else:
    (DeltaTable.forPath(spark, silver_path).alias("t")
        .merge(clean_df.alias("s"), "t.transaction_id = s.transaction_id")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())
    print("Table pehle se thi: MERGE ho gaya")

print("Silver rows:", spark.read.format("delta").load(silver_path).count())

Table pehle se thi: MERGE ho gaya
Silver rows: 100000


In [0]:
from pyspark.sql import functions as F

silver_df = spark.read.format("delta").load(silver_path)
total = silver_df.count()

checks = {
    "transaction_id null nahi": silver_df.filter(F.col("transaction_id").isNull()).count() == 0,
    "transaction_id unique": silver_df.select("transaction_id").distinct().count() == total,
    "amount null nahi": silver_df.filter(F.col("amount").isNull()).count() == 0,
    "amount zero nahi": silver_df.filter(F.col("amount") == 0).count() == 0,
    "currency valid (EUR/USD/QAR)": silver_df.filter(~F.col("currency").isin("EUR", "USD", "QAR")).count() == 0,
    "status valid": silver_df.filter(~F.col("status").isin("COMPLETED", "PENDING", "FAILED")).count() == 0,
    "date_key null nahi": silver_df.filter(F.col("date_key").isNull()).count() == 0,
    "PII columns hat gaye": not any(c in silver_df.columns for c in ["customer_name", "customer_email", "customer_id", "source_account_iban", "target_account_iban"]),
    "Bronze unique IDs = Silver rows (reconciliation)": raw_df.select("transaction_id").distinct().count() == total,
}

failed = [name for name, ok in checks.items() if not ok]

for name, ok in checks.items():
    print(("PASS  " if ok else "FAIL  ") + name)

print("\nSilver total rows:", total)

if failed:
    raise ValueError(f"Quality check fail hua: {failed}")
print("Sab checks pass. Silver data Gold ke liye ready hai.")

PASS  transaction_id null nahi
PASS  transaction_id unique
PASS  amount null nahi
PASS  amount zero nahi
PASS  currency valid (EUR/USD/QAR)
PASS  status valid
PASS  date_key null nahi
PASS  PII columns hat gaye
PASS  Bronze unique IDs = Silver rows (reconciliation)

Silver total rows: 100000
Sab checks pass. Silver data Gold ke liye ready hai.


In [0]:
from pyspark.sql import functions as F, Window

gold = f"abfss://gold@{ACCOUNT}.dfs.core.windows.net"
silver_df = spark.read.format("delta").load(silver_path)

# 1. dim_date: pehle se aakhri date tak ke saare din
bounds = silver_df.agg(F.min(F.to_date("transaction_ts")).alias("mn"),
                       F.max(F.to_date("transaction_ts")).alias("mx")).first()

dim_date = (
    spark.sql(f"SELECT explode(sequence(to_date('{bounds.mn}'), to_date('{bounds.mx}'), interval 1 day)) AS full_date")
    .withColumn("date_key", F.date_format("full_date", "yyyyMMdd").cast("int"))
    .withColumn("year", F.year("full_date"))
    .withColumn("month", F.month("full_date"))
    .withColumn("day_of_week", F.dayofweek("full_date"))
    .withColumn("is_weekend", F.dayofweek("full_date").isin(1, 7))
    .select("date_key", "full_date", "year", "month", "day_of_week", "is_weekend")
)

# 2. dim_customer: har customer ek row, volume ke hisaab se tier
cust = (silver_df.groupBy("customer_id_hashed")
        .agg(F.sum(F.abs("amount")).alias("total_volume"),
             F.min(F.to_date("transaction_ts")).alias("first_transaction_date")))

rank = F.percent_rank().over(Window.orderBy("total_volume"))
dim_customer = (cust
    .withColumn("customer_tier",
        F.when(rank >= 0.8, "PREMIUM").when(rank >= 0.4, "STANDARD").otherwise("BASIC"))
    .select("customer_id_hashed", "customer_tier", "first_transaction_date"))

# 3. fact_payments
fact_payments = silver_df.select(
    "transaction_id", "customer_id_hashed", "date_key",
    "amount", "currency", "direction", "category", "status")

# 4. agg_daily_cashflow: sirf COMPLETED transactions
agg_daily_cashflow = (fact_payments.filter(F.col("status") == "COMPLETED")
    .groupBy("date_key", "currency", "category")
    .agg(F.count("transaction_id").alias("total_count"),
         F.sum(F.when(F.col("direction") == "INFLOW", F.col("amount")).otherwise(0)).alias("total_inflow"),
         F.sum(F.when(F.col("direction") == "OUTFLOW", -F.col("amount")).otherwise(0)).alias("total_outflow")))

# Delta mein likho
for name, df in [("dim_date", dim_date), ("dim_customer", dim_customer),
                 ("fact_payments", fact_payments), ("agg_daily_cashflow", agg_daily_cashflow)]:
    df.write.format("delta").mode("overwrite").save(f"{gold}/{name}")
    print(name, "->", df.count(), "rows")

dim_date -> 22 rows


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


dim_customer -> 2000 rows
fact_payments -> 100000 rows
agg_daily_cashflow -> 264 rows


In [0]:
fact = spark.read.format("delta").load(f"{gold}/fact_payments")
agg = spark.read.format("delta").load(f"{gold}/agg_daily_cashflow")

fact_net = fact.filter("status = 'COMPLETED'").agg(F.sum("amount")).first()[0]
agg_net = agg.agg(F.sum(F.col("total_inflow") - F.col("total_outflow"))).first()[0]

print("Fact net (COMPLETED):", fact_net)
print("Agg net:", agg_net)
print("Match:", fact_net == agg_net)

Fact net (COMPLETED): 22955767.30
Agg net: 22955767.30
Match: True


In [0]:
display(spark.sql("SHOW CATALOGS"))
print("Current catalog:", spark.sql("SELECT current_catalog()").first()[0])

catalog
dbw_fintech_lakehouse
samples
system


Current catalog: dbw_fintech_lakehouse


In [0]:
spark.sql("USE CATALOG dbw_fintech_lakehouse")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

for name in ["dim_date", "dim_customer", "fact_payments", "agg_daily_cashflow"]:
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS dbw_fintech_lakehouse.gold.{name}
        USING DELTA
        LOCATION '{gold}/{name}'
    """)
    print("Registered:", name)

display(spark.sql("SHOW TABLES IN dbw_fintech_lakehouse.gold"))

Registered: dim_date
Registered: dim_customer
Registered: fact_payments
Registered: agg_daily_cashflow


database,tableName,isTemporary
gold,agg_daily_cashflow,false
gold,dim_customer,false
gold,dim_date,false
gold,fact_payments,false


In [0]:
display(spark.sql("""
    SELECT d.full_date, SUM(f.amount) AS net_amount, COUNT(*) AS txns
    FROM dbw_fintech_lakehouse.gold.fact_payments f
    JOIN dbw_fintech_lakehouse.gold.dim_date d ON f.date_key = d.date_key
    WHERE f.status = 'COMPLETED'
    GROUP BY d.full_date
    ORDER BY d.full_date
    LIMIT 10
"""))

full_date,net_amount,txns
2026-10-01,996892.70,2673
2026-10-02,1018760.05,2710
2026-10-03,1043904.31,2787
2026-10-04,1002834.14,2668
2026-10-05,1171330.39,2786
2026-10-06,985048.53,2638
2026-10-07,1012353.75,2634
2026-10-08,974220.38,2672
2026-10-09,911674.97,2751
2026-10-10,1101414.46,2756


In [0]:
fact = spark.read.format("delta").load(f"{gold}/fact_payments")
agg = spark.read.format("delta").load(f"{gold}/agg_daily_cashflow")

fact_net = fact.filter("status = 'COMPLETED'").agg(F.sum("amount")).first()[0]
agg_net = agg.agg(F.sum(F.col("total_inflow") - F.col("total_outflow"))).first()[0]

print("Fact net (COMPLETED):", fact_net)
print("Agg net:", agg_net)
print("Match:", fact_net == agg_net)


Fact net (COMPLETED): 22955767.30
Agg net: 22955767.30
Match: True
